# 🤖 Autonomous Feature Engineering with LLM Agents

> **"What if an AI could read the data description, understand feature relationships, and engineer new features—all while learning from its own mistakes?"**

---

## 🎬 Watch the Agent in Action

<!-- Replace YOUR_VIDEO_URL with your actual YouTube video URL -->

In [ ]:
from IPython.display import HTML, Image

# Embed video (replace with your actual video URL)
# HTML('<iframe width="100%" height="400" src="https://www.youtube.com/embed/YOUR_VIDEO_ID" frameborder="0" allowfullscreen></iframe>')
print("🎬 Video placeholder - replace with your YouTube embed")

---

# 💡 The Innovation

## The Problem

Feature engineering is often the most impactful yet tedious part of machine learning. Data scientists spend hours:
- Reading documentation to understand feature meanings
- Experimenting with transformations (log, square, ratios)
- Creating interaction features based on domain knowledge
- Testing each feature to see if it actually helps

**What if we could automate this entire process?**

## Our Solution: An Autonomous Feature Engineering Agent

We built an **LLM-powered agent** that:

| Innovation | Description |
|------------|-------------|
| 📄 **Reads Documentation** | Understands `data_description.txt` to learn what each column means |
| 📊 **Uses SHAP Feedback** | Analyzes which features matter most and targets improvements there |
| 🧪 **Self-Evaluates** | Tests each feature with 5-fold CV, keeps only what improves RMSLE |
| ⚡ **Runs in Parallel** | Producer-consumer pattern with batched Gemini API calls |
| 🧠 **Learns from Failure** | Remembers what didn't work to avoid repeating mistakes |

---

## Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                   AUTONOMOUS AGENT LOOP                          │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   📄 Data Description    ──────▶  🧠 Gemini LLM                 │
│   📊 SHAP Insights       ──────▶  (Feature Generator)           │
│   ❌ Failed Attempts     ──────▶                                │
│                                        │                        │
│                                        ▼                        │
│                               ┌──────────────┐                  │
│                               │  Python Code │                  │
│                               │  Generator   │                  │
│                               └──────┬───────┘                  │
│                                      │                          │
│                                      ▼                          │
│   ┌──────────────┐           ┌──────────────┐                  │
│   │   LightGBM   │◀──────────│   Sandbox    │                  │
│   │   5-Fold CV  │           │   Executor   │                  │
│   └──────┬───────┘           └──────────────┘                  │
│          │                                                      │
│          ▼                                                      │
│   ┌──────────────┐           ┌──────────────┐                  │
│   │  RMSLE       │──────────▶│   Memory     │                  │
│   │  Improved?   │           │   (JSON)     │                  │
│   └──────────────┘           └──────────────┘                  │
│          │                                                      │
│    ✅ KEEP  /  ❌ REJECT                                        │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

# 📊 Real-Time Dashboard

We built a **real-time monitoring dashboard** that shows the agent's progress:

- **Worker Status**: See parallel workers evaluating features simultaneously
- **Feature Queue**: Producer-consumer pattern in action
- **Live RMSLE Chart**: Watch the score improve in real-time
- **Accept/Reject Feed**: Instant feedback on each feature

<!-- Replace with your dashboard screenshot -->

In [ ]:
# Dashboard screenshot placeholder
# Image('dashboard_screenshot.png', width=800)
print("📊 Dashboard screenshot placeholder - add your screenshot here")

---

# 🏆 Agent Results

## Performance Summary

| Metric | Value |
|--------|-------|
| **Baseline RMSLE** | 0.10252 |
| **Best RMSLE** | 0.10153 |
| **Improvement** | 0.97% |
| **Features Tried** | 10 |
| **Features Kept** | 3 (30% success rate) |

## Features Discovered by the Agent

| # | Feature | Strategy | Description | RMSLE Improvement |
|---|---------|----------|-------------|-------------------|
| 1 | `Neighborhood_Freq` | Frequency Encoding | Count of houses in each neighborhood | +0.00030 |
| 2 | `TotalBathrooms` | Aggregation | Full + 0.5×Half + Basement baths | +0.00002 |
| 3 | `Has2ndFloor` | Binary Indicator | Whether house has a second floor | +0.00068 |

## Progress Visualization

The chart below shows the agent's journey through 10 iterations:

In [ ]:
# Progress plot placeholder
# If running locally, uncomment:
# Image('../outputs/logs/progress_plot.png', width=800)
print("📈 Progress plot placeholder - the agent's RMSLE improvement over iterations")

## Sample Agent Reasoning

Here's an example of how the agent thinks:

### Iteration 9: `Has2ndFloor` (✅ KEPT)

**Agent's Input:**
- SHAP showed `2ndFlrSF` had moderate importance
- Many houses have 0 for 2ndFlrSF (no second floor)

**Agent's Reasoning:**
> "Create a binary indicator to capture the categorical distinction between houses with and without second floors."

**Generated Code:**
```python
df['Has2ndFloor'] = (df['2ndFlrSF'].fillna(0) > 0).astype(int)
```

**Result:** RMSLE improved from 0.10221 → 0.10153 (**+0.00068**)

---

# 💻 Reproducible Code

Below is the complete, reproducible pipeline. Judges can run this end-to-end to verify our results.

## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_log_error, mean_squared_error
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print('✅ Imports successful!')

In [ ]:
def rmsle(y_true, y_pred):
    """Calculate Root Mean Squared Logarithmic Error"""
    return np.sqrt(mean_squared_log_error(y_true, y_pred))

def rmse(y_true, y_pred):
    """Calculate Root Mean Squared Error"""
    return np.sqrt(mean_squared_error(y_true, y_pred))

print('✅ Helper functions defined!')

## 2. Load Data

In [ ]:
import os

# Environment detection: Kaggle or local
if os.path.exists('/kaggle/input'):
    DATA_PATH = '/kaggle/input/house-prices-advanced-regression-techniques'
    OUTPUT_PATH = '/kaggle/working'
    print('🏠 Running in Kaggle environment')
else:
    DATA_PATH = '../data' if os.path.exists('../data/train.csv') else 'data'
    OUTPUT_PATH = '../outputs/predictions' if os.path.exists('../outputs') else 'outputs/predictions'
    print('💻 Running in local environment')

os.makedirs(OUTPUT_PATH, exist_ok=True)

train = pd.read_csv(f'{DATA_PATH}/train.csv')
test = pd.read_csv(f'{DATA_PATH}/test.csv')

print(f'\n📊 Train shape: {train.shape}')
print(f'📊 Test shape: {test.shape}')

## 3. Preprocessing Class

In [ ]:
class AmesPreprocessor:
    """
    Comprehensive preprocessing for Ames Housing data
    
    Features:
    - Semantic imputation (NA meanings)
    - Grouped imputation (LotFrontage by Neighborhood)
    - Ordinal encoding for quality features
    - One-hot encoding for categorical features
    - Engineered features (TotalSF, HouseAge, etc.)
    """
    
    def __init__(self):
        self.numeric_impute_values = {}
        self.neighborhood_lot_frontage = {}
        self.feature_columns = None
        self.fitted = False
    
    def fit_transform(self, df):
        df = df.copy()
        df = self._create_engineered_features(df)
        df = self._handle_missing_values(df, fit=True)
        df = self._encode_quality_features(df)
        df = self._encode_categorical_features(df)
        self.feature_columns = df.columns.tolist()
        self.fitted = True
        return df
    
    def transform(self, df):
        if not self.fitted:
            raise ValueError("Preprocessor must be fitted before transform")
        df = df.copy()
        df = self._create_engineered_features(df)
        df = self._handle_missing_values(df, fit=False)
        df = self._encode_quality_features(df)
        df = self._encode_categorical_features(df)
        
        for col in self.feature_columns:
            if col not in df.columns:
                df[col] = 0
        df = df[self.feature_columns]
        return df
    
    def _create_engineered_features(self, df):
        df['TotalSF'] = df['TotalBsmtSF'].fillna(0) + df['1stFlrSF'].fillna(0) + df['2ndFlrSF'].fillna(0)
        df['HouseAge'] = df['YrSold'] - df['YearBuilt']
        df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']
        df['TotalBath'] = df['FullBath'].fillna(0) + 0.5*df['HalfBath'].fillna(0) + df['BsmtFullBath'].fillna(0) + 0.5*df['BsmtHalfBath'].fillna(0)
        df['PorchArea'] = df['OpenPorchSF'].fillna(0) + df['EnclosedPorch'].fillna(0) + df['3SsnPorch'].fillna(0) + df['ScreenPorch'].fillna(0)
        return df
    
    def _handle_missing_values(self, df, fit=False):
        na_means_none = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
                         'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                         'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'MasVnrType']
        for col in na_means_none:
            if col in df.columns:
                df[col] = df[col].fillna('None')
        
        na_means_zero = ['GarageYrBlt', 'GarageArea', 'GarageCars',
                         'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
                         'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']
        for col in na_means_zero:
            if col in df.columns:
                df[col] = df[col].fillna(0)
        
        if 'LotFrontage' in df.columns:
            if fit:
                self.neighborhood_lot_frontage = df.groupby('Neighborhood')['LotFrontage'].median().to_dict()
            df['LotFrontage'] = df.apply(
                lambda row: self.neighborhood_lot_frontage.get(row['Neighborhood'], 0)
                if pd.isna(row['LotFrontage']) else row['LotFrontage'], axis=1)
        
        numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
        if fit:
            for col in numeric_cols:
                if df[col].isna().sum() > 0:
                    self.numeric_impute_values[col] = df[col].median()
        for col, val in self.numeric_impute_values.items():
            if col in df.columns:
                df[col] = df[col].fillna(val)
        
        categorical_cols = df.select_dtypes(include=['object']).columns
        for col in categorical_cols:
            if df[col].isna().sum() > 0:
                df[col] = df[col].fillna(df[col].mode()[0] if len(df[col].mode()) > 0 else 'None')
        return df
    
    def _encode_quality_features(self, df):
        qual_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
        qual_features = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
                         'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']
        for col in qual_features:
            if col in df.columns:
                df[col] = df[col].map(qual_map).fillna(0).astype(int)
        
        if 'BsmtExposure' in df.columns:
            df['BsmtExposure'] = df['BsmtExposure'].map({'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0}).fillna(0).astype(int)
        
        bsmt_fin_map = {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'None': 0}
        if 'BsmtFinType1' in df.columns:
            df['BsmtFinType1'] = df['BsmtFinType1'].map(bsmt_fin_map).fillna(0).astype(int)
        if 'BsmtFinType2' in df.columns:
            df['BsmtFinType2'] = df['BsmtFinType2'].map(bsmt_fin_map).fillna(0).astype(int)
        
        if 'Functional' in df.columns:
            df['Functional'] = df['Functional'].map({'Typ': 7, 'Min1': 6, 'Min2': 5, 'Mod': 4, 'Maj1': 3, 'Maj2': 2, 'Sev': 1, 'Sal': 0}).fillna(7).astype(int)
        
        if 'GarageFinish' in df.columns:
            df['GarageFinish'] = df['GarageFinish'].map({'Fin': 3, 'RFn': 2, 'Unf': 1, 'None': 0}).fillna(0).astype(int)
        return df
    
    def _encode_categorical_features(self, df):
        categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
        if len(categorical_cols) > 0:
            df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
        return df

print('✅ AmesPreprocessor class defined!')

## 4. Agent-Discovered Features

These features were **automatically discovered** by our LLM agent. Each feature includes comments explaining when and how it was found.

In [ ]:
def apply_agent_features(df):
    """
    Features discovered by the Autonomous Feature Engineering Agent.
    
    These features were generated by Gemini LLM, tested with 5-fold CV,
    and kept only because they improved RMSLE.
    """
    df = df.copy()
    
    # Feature 1: Neighborhood_Freq
    # Strategy: Frequency Encoding
    # Discovered: Iteration 1
    # Improvement: +0.00030 RMSLE
    # Reasoning: Houses in popular neighborhoods may have different value patterns
    df['Neighborhood_Freq'] = df.groupby('Neighborhood')['Neighborhood'].transform('count').fillna(0)
    
    # Feature 2: TotalBathrooms
    # Strategy: Aggregation/Interaction
    # Discovered: Iteration 7
    # Improvement: +0.00002 RMSLE
    # Reasoning: Combines all bathroom types with appropriate weighting
    df['TotalBathrooms'] = (df['FullBath'].fillna(0) + 
                           0.5 * df['HalfBath'].fillna(0) + 
                           df['BsmtFullBath'].fillna(0) + 
                           0.5 * df['BsmtHalfBath'].fillna(0))
    
    # Feature 3: Has2ndFloor
    # Strategy: Binary Indicator
    # Discovered: Iteration 9
    # Improvement: +0.00068 RMSLE (best improvement!)
    # Reasoning: Categorical distinction for multi-story vs single-story homes
    df['Has2ndFloor'] = (df['2ndFlrSF'].fillna(0) > 0).astype(int)
    
    return df

print('✅ Agent-discovered features defined!')
print('   - Neighborhood_Freq (frequency encoding)')
print('   - TotalBathrooms (aggregation)')
print('   - Has2ndFloor (binary indicator)')

## 5. Preprocessing Pipeline

In [ ]:
# Extract target and log-transform
target = train['SalePrice']
target_log = np.log1p(target)
train_ids = train['Id']
test_ids = test['Id']

# Apply agent features BEFORE preprocessing (on raw data)
print('🤖 Applying agent-discovered features...')
train_with_agent = apply_agent_features(train)
test_with_agent = apply_agent_features(test)

# Preprocessing
print('🔧 Preprocessing data...')
preprocessor = AmesPreprocessor()
X_train_raw = train_with_agent.drop(['Id', 'SalePrice'], axis=1)
X_test_raw = test_with_agent.drop(['Id'], axis=1)

X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

print(f'\n📊 Training data shape: {X_train.shape}')
print(f'📊 Test data shape: {X_test.shape}')
print(f'\n✅ Total features: {X_train.shape[1]}')

## 6. Model Training (LightGBM with Optuna-tuned params)

In [ ]:
# Optuna-tuned hyperparameters
model_params = {
    'n_estimators': 800,
    'learning_rate': 0.04435737213089749,
    'max_depth': 13,
    'num_leaves': 69,
    'min_child_samples': 35,
    'subsample': 0.6173983977980623,
    'colsample_bytree': 0.6241701198264097,
    'reg_alpha': 0.015936847132187043,
    'reg_lambda': 6.573787083889792e-08,
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'random_state': 42
}

print('⚙️ Model parameters loaded (Optuna-tuned)')

In [ ]:
print('🔄 Running 5-Fold Cross-Validation...\n')

n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), 1):
    X_train_fold = X_train.iloc[train_idx]
    X_val_fold = X_train.iloc[val_idx]
    y_train_fold = target_log.iloc[train_idx]
    y_val_fold = target_log.iloc[val_idx]
    y_val_original = target.iloc[val_idx]
    
    model = lgb.LGBMRegressor(**model_params)
    model.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    val_pred = np.expm1(model.predict(X_val_fold))
    fold_rmsle = rmsle(y_val_original, val_pred)
    cv_scores.append(fold_rmsle)
    print(f'   Fold {fold}: RMSLE = {fold_rmsle:.5f}')

mean_cv = np.mean(cv_scores)
std_cv = np.std(cv_scores)

print('\n' + '='*50)
print(f'📈 Mean CV RMSLE: {mean_cv:.5f} (± {std_cv:.5f})')
print('='*50)

## 7. Final Model & Predictions

In [ ]:
print('🏋️ Training final model on full dataset...')

final_model = lgb.LGBMRegressor(**model_params)
final_model.fit(X_train, target_log)

print('🎯 Making predictions on test set...')
test_pred = np.expm1(final_model.predict(X_test))

print(f'\n📊 Prediction statistics:')
print(f'   Mean: ${test_pred.mean():,.2f}')
print(f'   Median: ${np.median(test_pred):,.2f}')
print(f'   Min: ${test_pred.min():,.2f}')
print(f'   Max: ${test_pred.max():,.2f}')

## 8. Create Submission

In [ ]:
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': test_pred
})

submission.to_csv(f'{OUTPUT_PATH}/submission.csv', index=False)

print(f'💾 Submission saved to {OUTPUT_PATH}/submission.csv')
print(f'\n📋 Submission preview:')
submission.head(10)

---

# 🎯 Conclusion

## What We Achieved

- **Built an autonomous agent** that reads documentation and engineers features without human intervention
- **Discovered 3 impactful features** through automated experimentation
- **Improved RMSLE by ~1%** through agent-discovered features
- **Created a real-time dashboard** for monitoring the agent's progress

## Technical Innovations

1. **SHAP-Guided Feature Generation**: The agent uses SHAP values to understand which features matter most, then generates features that interact with or enhance them.

2. **Self-Evaluation Loop**: Every generated feature is tested with 5-fold CV. Only improvements are kept.

3. **Parallel Execution**: Producer-consumer pattern allows efficient batching of Gemini API calls while workers evaluate features in parallel.

4. **Memory & Learning**: The agent remembers what worked and what didn't, avoiding duplicate attempts.

## Future Directions

- **Multi-agent collaboration**: Multiple agents with different strategies
- **More feature strategies**: Time-series patterns, clustering-based features
- **AutoML integration**: Combine with hyperparameter optimization
- **Cross-competition transfer**: Learn feature patterns that work across Kaggle competitions

---

## 📎 Resources

- **GitHub**: [Link to repository]
- **Video Demo**: [Link to YouTube video]
- **Architecture Docs**: [Link to documentation]

---

*Built with ❤️ using Gemini LLM, LightGBM, and Python*